# SpaceX Capstone — Predictive Analysis (Classification)

Predict first-stage landing outcome (`Class`: 1=successful, 0=unsuccessful). The feature set excludes the target and several post-outcome/identifier fields to reduce leakage.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
URL='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv'
df=pd.read_csv(URL)
df.head()

In [ ]:
drop_cols=[c for c in ['Class','Date','Outcome','LandingPad','Serial','BoosterVersion'] if c in df.columns]
X=df.drop(columns=drop_cols).copy(); y=df['Class'].astype(int)
X=pd.get_dummies(X,dummy_na=True)
X=X.replace([float('inf'),float('-inf')],pd.NA).fillna(X.median(numeric_only=True)).fillna(0)
X=X.loc[:,X.nunique(dropna=False)>1]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
X.shape, y.value_counts(normalize=True)

In [ ]:
models={
 'Logistic Regression':Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=3000,class_weight='balanced'))]),
 'SVM':Pipeline([('scale',StandardScaler()),('model',SVC(class_weight='balanced'))]),
 'Decision Tree':DecisionTreeClassifier(random_state=42,class_weight='balanced'),
 'KNN':Pipeline([('scale',StandardScaler()),('model',KNeighborsClassifier(n_neighbors=5))])}
rows=[]; fitted={}
for name,model in models.items():
 model.fit(X_train,y_train); pred=model.predict(X_test); fitted[name]=model
 rows.append({'model':name,'accuracy':accuracy_score(y_test,pred),'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred,zero_division=0),'f1':f1_score(y_test,pred,zero_division=0)})
metrics=pd.DataFrame(rows).sort_values('f1',ascending=False)
metrics

In [ ]:
best_name=metrics.iloc[0]['model']; best=fitted[best_name]; pred=best.predict(X_test)
print('Best model by F1:',best_name)
print(classification_report(y_test,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred); plt.title('Best model confusion matrix'); plt.tight_layout()

In [ ]:
metrics.set_index('model')[['accuracy','precision','recall','f1']].plot(kind='bar',ylim=(0,1),figsize=(9,4),title='Model comparison')
plt.ylabel('Score'); plt.xticks(rotation=20); plt.tight_layout()

## Model selection and limitations

Select the model using a stated metric (F1 is useful when both false positives and false negatives matter), then inspect its confusion matrix. Results depend on the fixed split and the small teaching dataset. For a final report, report the measured scores shown by this notebook and explain the business cost of missed landing failures.